## Solving CartPole with DeepQNetworks


In [1]:
%load_ext tensorboard

TODO: I plan to write a tutorial on going through how DQN's work, and how it compares to other algorithms like policy gradient methods, and actor critic methods. Deep Q networks solved the stability issue by introducing target networks and replay buffers both of which have interesting extensions that can be extended into the samsara rl library due to its use of the composition pattern. 

Deep Q networks first learn the true values of states adjacent to terminal states. These are points the target network which is initially noise is zero'd out (no future reward) and only the R signal computes to the temporal difference. 

In future iterations, the bootstrap values from the adjacent states propogate to the rest of the state space but require careful tuning, gradient clipping to combat noisy bootstrap estimates. Batch learning  helps reduce variance and move towards stable learning.

Until I get to the full tutorial of how deep q nets work this full Gym example below shows how to use the Deep Q Network in the library. It is capable of running any gym environment, here I used cart pole

In [2]:
import os

import gymnasium as gym
import structlog
from cart_pole_logger import CartPoleLogger

# from samsara_rl.mdp.terminal_penalty_wrapper import TerminalPenaltyWrapper
from samsara_rl.control.function_approximation.batch.deep_q_network.q_network import QNetwork
from samsara_rl.control.function_approximation.functions.neural_networks.fully_connected import (
    FullyConnected,
)

from samsara_rl.mdp.cart_pole.scaled_cart_pole import ScaledCartPole

log = structlog.get_logger()

In [3]:
log = structlog.get_logger()

MAX_EPISODES = 2800
EVAL_EPISODES = 20
BASE_DIR = "logs/double_dqn_example"
ALPHAS = [0.0001, 0.0003, 0.001]  # 0.005, 0.001, 0.0005, 0.0001, 0.00005]
GAMMAS = [0.99]

In [7]:
def run_experiment(env, alpha: float, gamma: float) -> None:
    """Train and evaluate a single configuration.

    Creates a LinearFunction and QLearningGradient agent, trains for
    MAX_EPISODES episodes, saves the learned weights, and runs greedy
    evaluation.

    Args:
        env: Gymnasium CartPole environment.
        alpha: Learning rate for the Q-learning update.
    """
    model_name = "deep_q_network"
    run_name = f"{model_name}_alpha={alpha}_gamma={gamma}"
    run_dir = os.path.join(BASE_DIR, run_name)
    os.makedirs(run_dir, exist_ok=True)

    log.info("starting_run", run=run_name)

    fc = FullyConnected(4, 32, 2, preprocess=None)

    agent = QNetwork(mdp=env, gamma=gamma, q=fc, alpha=alpha, log_dir=run_dir, experiment_name=run_name)

    agent.register(CartPoleLogger())

    agent.evaluate(max_iter=MAX_EPISODES)
    #     save_model(agent, run_dir)

    if agent.tensorboard:
        agent.tensorboard.flush()

In [8]:
def main() -> None:
    """Run grid search over feature configs, bias, and learning rates.

    Iterates over all combinations of FEATURE_CONFIGS, BIAS_CONFIGS,
    and ALPHAS, training and evaluating each one.
    """
    env = ScaledCartPole(gym.make("CartPole-v1"))

    for alpha in ALPHAS:
        for gamma in GAMMAS:
            run_experiment(env, alpha, gamma)

In [9]:
main()

2026-09-12 12:53:10 [info     ] starting_run                   run='deep_q_network_alpha=0.0001_gamma=0.99'


/home/ashish/reinforcement-learning-library/samsara-rl/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:979: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


2026-09-12 12:53:22 [debug    ] target_network_swap            episode=68
2026-09-12 12:53:29 [debug    ] target_network_swap            episode=114
2026-09-12 12:53:38 [debug    ] target_network_swap            episode=161
2026-09-12 12:53:45 [debug    ] target_network_swap            episode=209
2026-09-12 12:53:51 [debug    ] target_network_swap            episode=259
2026-09-12 12:54:00 [debug    ] target_network_swap            episode=302
2026-09-12 12:54:07 [debug    ] target_network_swap            episode=354
2026-09-12 12:54:15 [debug    ] target_network_swap            episode=416
2026-09-12 12:54:23 [debug    ] target_network_swap            episode=468
2026-09-12 12:54:32 [debug    ] target_network_swap            episode=516
2026-09-12 12:54:40 [debug    ] target_network_swap            episode=577
2026-09-12 12:54:47 [debug    ] target_network_swap            episode=610
2026-09-12 12:54:52 [debug    ] target_network_swap            episode=636
2026-09-12 12:54:57 [debug

2026-09-12 12:59:03 [debug    ] target_network_swap            episode=1016
2026-09-12 12:59:07 [debug    ] target_network_swap            episode=1019
2026-09-12 12:59:13 [debug    ] target_network_swap            episode=1024
2026-09-12 12:59:17 [debug    ] target_network_swap            episode=1029
2026-09-12 12:59:21 [debug    ] target_network_swap            episode=1035
2026-09-12 12:59:24 [debug    ] target_network_swap            episode=1039
2026-09-12 12:59:29 [debug    ] target_network_swap            episode=1047
2026-09-12 12:59:33 [debug    ] target_network_swap            episode=1054
2026-09-12 12:59:37 [debug    ] target_network_swap            episode=1058
2026-09-12 12:59:42 [debug    ] target_network_swap            episode=1064
2026-09-12 12:59:46 [debug    ] target_network_swap            episode=1070
2026-09-12 12:59:50 [debug    ] target_network_swap            episode=1076
2026-09-12 12:59:55 [debug    ] target_network_swap            episode=1083
2026-09-12 1

2026-09-12 13:03:44 [debug    ] target_network_swap            episode=1302
2026-09-12 13:03:48 [debug    ] target_network_swap            episode=1304
2026-09-12 13:03:52 [debug    ] target_network_swap            episode=1306
2026-09-12 13:03:56 [debug    ] target_network_swap            episode=1308
2026-09-12 13:04:00 [debug    ] target_network_swap            episode=1310
2026-09-12 13:04:06 [debug    ] target_network_swap            episode=1312
2026-09-12 13:04:10 [debug    ] target_network_swap            episode=1315
2026-09-12 13:04:14 [debug    ] target_network_swap            episode=1318
2026-09-12 13:04:19 [debug    ] target_network_swap            episode=1320
2026-09-12 13:04:23 [debug    ] target_network_swap            episode=1323
2026-09-12 13:04:27 [debug    ] target_network_swap            episode=1325
2026-09-12 13:04:31 [debug    ] target_network_swap            episode=1327
2026-09-12 13:04:35 [debug    ] target_network_swap            episode=1329
2026-09-12 1

...

2026-09-12 13:59:27 [debug    ] target_network_swap            episode=2538
2026-09-12 13:59:37 [debug    ] target_network_swap            episode=2540
2026-09-12 13:59:44 [debug    ] target_network_swap            episode=2542
2026-09-12 13:59:52 [debug    ] target_network_swap            episode=2544
2026-09-12 13:59:59 [debug    ] target_network_swap            episode=2546
2026-09-12 14:00:07 [debug    ] target_network_swap            episode=2548
2026-09-12 14:00:14 [debug    ] target_network_swap            episode=2550
2026-09-12 14:00:22 [debug    ] target_network_swap            episode=2552
2026-09-12 14:00:29 [debug    ] target_network_swap            episode=2554
2026-09-12 14:00:37 [debug    ] target_network_swap            episode=2556
2026-09-12 14:00:44 [debug    ] target_network_swap            episode=2558
2026-09-12 14:00:54 [debug    ] target_network_swap            episode=2560
2026-09-12 14:01:02 [debug    ] target_network_swap            episode=2562
2026-09-12 1

2026-09-12 14:08:59 [debug    ] target_network_swap            episode=2760
2026-09-12 14:09:06 [debug    ] target_network_swap            episode=2772
2026-09-12 14:09:15 [debug    ] target_network_swap            episode=2781
2026-09-12 14:09:22 [debug    ] target_network_swap            episode=2790
2026-09-12 14:09:28 [debug    ] target_network_swap            episode=2798
2026-09-12 14:09:31 [info     ] starting_run                   run='deep_q_network_alpha=0.001_gamma=0.99'
2026-09-12 14:09:47 [debug    ] target_network_swap            episode=69
2026-09-12 14:09:58 [debug    ] target_network_swap            episode=110
2026-09-12 14:10:09 [debug    ] target_network_swap            episode=149
2026-09-12 14:10:18 [debug    ] target_network_swap            episode=176
2026-09-12 14:10:29 [debug    ] target_network_swap            episode=204
2026-09-12 14:10:38 [debug    ] target_network_swap            episode=233
2026-09-12 14:10:47 [debug    ] target_network_swap            e

2026-09-12 14:18:29 [debug    ] target_network_swap            episode=1106
2026-09-12 14:18:38 [debug    ] target_network_swap            episode=1120
2026-09-12 14:18:45 [debug    ] target_network_swap            episode=1135
2026-09-12 14:18:53 [debug    ] target_network_swap            episode=1149
2026-09-12 14:19:02 [debug    ] target_network_swap            episode=1163
2026-09-12 14:19:09 [debug    ] target_network_swap            episode=1177
2026-09-12 14:19:18 [debug    ] target_network_swap            episode=1189
2026-09-12 14:19:27 [debug    ] target_network_swap            episode=1200
2026-09-12 14:19:33 [debug    ] target_network_swap            episode=1218
2026-09-12 14:19:42 [debug    ] target_network_swap            episode=1234
2026-09-12 14:19:51 [debug    ] target_network_swap            episode=1245
2026-09-12 14:19:57 [debug    ] target_network_swap            episode=1257
2026-09-12 14:20:06 [debug    ] target_network_swap            episode=1267
2026-09-12 1

2026-09-12 14:27:29 [debug    ] target_network_swap            episode=1954
2026-09-12 14:27:37 [debug    ] target_network_swap            episode=1965
2026-09-12 14:27:44 [debug    ] target_network_swap            episode=1976
2026-09-12 14:27:53 [debug    ] target_network_swap            episode=1989
2026-09-12 14:28:02 [debug    ] target_network_swap            episode=2004
2026-09-12 14:28:08 [debug    ] target_network_swap            episode=2016
2026-09-12 14:28:17 [debug    ] target_network_swap            episode=2031
2026-09-12 14:28:26 [debug    ] target_network_swap            episode=2043
2026-09-12 14:28:32 [debug    ] target_network_swap            episode=2055
2026-09-12 14:28:41 [debug    ] target_network_swap            episode=2067
2026-09-12 14:28:50 [debug    ] target_network_swap            episode=2081
2026-09-12 14:28:56 [debug    ] target_network_swap            episode=2096
2026-09-12 14:29:05 [debug    ] target_network_swap            episode=2107
2026-09-12 1

2026-09-12 14:35:02 [debug    ] target_network_swap            episode=2462
2026-09-12 14:35:07 [debug    ] target_network_swap            episode=2474
2026-09-12 14:35:12 [debug    ] target_network_swap            episode=2484
2026-09-12 14:35:16 [debug    ] target_network_swap            episode=2493
2026-09-12 14:35:22 [debug    ] target_network_swap            episode=2501
2026-09-12 14:35:26 [debug    ] target_network_swap            episode=2507
2026-09-12 14:35:31 [debug    ] target_network_swap            episode=2513
2026-09-12 14:35:35 [debug    ] target_network_swap            episode=2519
2026-09-12 14:35:41 [debug    ] target_network_swap            episode=2525
2026-09-12 14:35:45 [debug    ] target_network_swap            episode=2530
2026-09-12 14:35:49 [debug    ] target_network_swap            episode=2535
2026-09-12 14:35:55 [debug    ] target_network_swap            episode=2541
2026-09-12 14:35:59 [debug    ] target_network_swap            episode=2546
2026-09-12 1